# Task 2.1 

In [3]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/hdt7 ms')

Mounted at /content/drive


In [ ]:

import math
g = 3                                  
mu = 0.25
lam = 3.0 + 0.2 * ((g - 1) % 5)
c = 17 + ((g - 1) % 3)
a = lam / mu
rho = a / c
k_off = math.ceil(c - a) + 3
Tb = 7
print(f"Grupo {g}: lambda={lam:.1f}/día, c={c}, a={a:.2f}, rho={rho:.4f}, k_off={k_off}, Tb={Tb} días")

import numpy as np

def erlang_b(c, a):
    """Recurrencia: B(0)=1;  B(k) = a B(k-1) / (k + a B(k-1))"""
    B = 1.0
    for k in range(1, c + 1):
        B = a * B / (k + a * B)
    return B

def erlang_c_direct(c, a):
    """C(c,a) = [a^c/c! /(1-rho)] / [sum_{k<c} a^k/k! + a^c/c! /(1-rho)]  (estable vía logs)"""
    from math import lgamma, log, exp
    rho = a / c
    terms = [exp(k*log(a) - lgamma(k+1)) for k in range(c)]
    top = exp(c*log(a) - lgamma(c+1)) / (1 - rho)
    return top / (sum(terms) + top)

def theory_wait(lam, mu, c):
    a = lam / mu; rho = a / c
    assert rho < 1
    B = erlang_b(c, a)
    C = B / (1 - rho * (1 - B))             # Erlang C a partir de Erlang B
    Lq = C * rho / (1 - rho)
    Wq = Lq / lam                            # = C/(c mu - lam)
    W = Wq + 1 / mu
    L = Lq + a
    return dict(a=a, rho=rho, C=C, Lq=Lq, Wq_h=Wq*24, W_h=W*24, Wq=Wq, W=W, L=L, util=rho, Pwait=C)

def theory_loss(lam, mu, c):
    a = lam / mu
    B = erlang_b(c, a)
    lam_eff = lam * (1 - B)
    L = lam_eff / mu                         # = a (1-B)
    return dict(B=B, lam_eff=lam_eff, L=L, util=L / c, W=1/mu, Pblock=B)


Grupo 3: lambda=3.4/día, c=19, a=13.60, rho=0.7158, k_off=9, Tb=7 días


## 2.1b — Tiempo de vaciado tras el cierre (modelo de acumulación)
Durante el cierre la capacidad es $\mu(c-k_{off})$. Si $\lambda>\mu(c-k_{off})$ la fila crece a tasa $\lambda-\mu(c-k_{off})$ durante $T_b$:
$$Q_b=T_b\,[\lambda-\mu(c-k_{off})].$$
Tras reabrir, la fila baja a la tasa que sobra, $\mu c-\lambda$:
$$\boxed{T_d=\frac{Q_b}{\mu c-\lambda}=T_b\,\frac{\lambda-\mu(c-k_{off})}{\mu c-\lambda}}$$


## 2.1c — Estancia +25 %
$\mu'=0.25/1.25=0.2$, $a'=\lambda/\mu'=17.0$ con $c=19$. Se evalúa la recurrencia de Erlang B con $a'=17$. Esperado: el bloqueo crece **mucho más que 25 %**, por la no linealidad de $B$ cerca de $\rho'=0.89$.

In [5]:
import json, datetime
Wq_mmc_h = theory_wait(lam, mu, c)['Wq_h']
AC = {cs: Wq_mmc_h * (1 + cs**2) / 2 for cs in (0, 1, 2)}
Qb = Tb * (lam - mu * (c - k_off)); Td = Qb / (mu * c - lam)
B0 = erlang_b(c, a); mu2 = mu / 1.25; a2 = lam / mu2; B1 = erlang_b(c, a2)
print("2.1a  AC  Wq [h]:", {k: round(v, 3) for k, v in AC.items()})
print(f"2.1b  Qb = {Qb:.2f} pacientes   Td = {Td:.3f} días = {Td*24:.1f} h")
print(f"2.1c  a'={a2:.1f}  B'={B1:.5f}  B actual={B0:.5f}  razón={B1/B0:.2f}x")
pred = dict(timestamp=datetime.datetime.now().isoformat(timespec="seconds"),
            AC_Wq_h={str(k): v for k, v in AC.items()}, Qb=Qb, Td_dias=Td, Td_sim_pred_dias=6.5,
            sim_Wq_pred_h={"0": 1.3, "1": Wq_mmc_h, "2": 3.5}, err_AC_pred={"0": -0.17, "1": 0.0, "2": 0.5},
            B_new=B1, B_old=B0, ratio=B1/B0)
json.dump(pred, open("predicciones.json", "w"), indent=1); print("Guardado predicciones.json", pred["timestamp"])

2.1a  AC  Wq [h]: {0: 1.07, 1: 2.14, 2: 5.349}
2.1b  Qb = 6.30 pacientes   Td = 4.667 días = 112.0 h
2.1c  a'=17.0  B'=0.11050  B actual=0.03743  razón=2.95x
Guardado predicciones.json 2026-10-15T17:52:41
